In [1]:
from src.metric import TimestampMetric, get_phase
from src.helpers import normalize_project, get_zip_json_data, iterate_parallel
import numpy as np

In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/code-results-9-4-2026"
OUTPUT_FOLDER = "../results"

In [3]:
def find_measurement(data: dict, group: str, name: str) -> str | None:
    for measurement in data["measurements"]:
        if measurement[0] == group:
            value = measurement[1].get(name)
            if value is None:
                print(f"Warning: measurement '{name}' not found in group '{group}'")
            return value
    return None

In [5]:
metrics = TimestampMetric("rca_unsafety_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_loc = 0
    other_loc = 0
    rust_unsafe_loc = 0
    other_unsafe_loc = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            rust_unsafe_loc += int(find_measurement(entry, "THESIS", "unsafe_lines") or 0)
        else:
            other_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            other_unsafe_loc += int(find_measurement(entry, "THESIS", "unsafe_lines") or 0)
    
    return [
        (normalize_project(project_name), timestamp, rust_unsafe_loc / rust_loc if rust_loc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_unsafe_loc / other_loc if other_loc > 0 else 0, False),
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:01<00:00, 24.63it/s]


In [6]:
metrics = TimestampMetric("rca_documentation_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_loc = 0
    other_loc = 0
    rust_cloc = 0
    other_cloc = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            rust_cloc += int(find_measurement(entry, "LOC", "cloc") or 0)
        else:
            other_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            other_cloc += int(find_measurement(entry, "LOC", "cloc") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_cloc / rust_loc if rust_loc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_cloc / other_loc if other_loc > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:00<00:00, 24.68it/s]


In [7]:
metrics = TimestampMetric("rca_cyclomatic_complexity", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_value = 0
    other_value = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_value += int(find_measurement(entry, "CYCLOMATIC", "sum") or 0)
        else:
            other_value += int(find_measurement(entry, "CYCLOMATIC", "sum") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_value, True),
        (normalize_project(project_name), timestamp, other_value, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:33<00:00, 23.12it/s]


In [8]:
metrics = TimestampMetric("rca_cognitive_complexity", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_value = 0
    other_value = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_value += int(find_measurement(entry, "COGNITIVE", "sum") or 0)
        else:
            other_value += int(find_measurement(entry, "COGNITIVE", "sum") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_value, True),
        (normalize_project(project_name), timestamp, other_value, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:34<00:00, 23.04it/s]


In [9]:
metrics = TimestampMetric("rca_cyclomatic_complexity_per_unit", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_values = []
    other_values = []

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "FUNCTION":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_values.append(int(find_measurement(entry, "CYCLOMATIC", "sum") or 0))
        else:
            other_values.append(int(find_measurement(entry, "CYCLOMATIC", "sum") or 0))

    return [
        (normalize_project(project_name), timestamp, np.mean(rust_values) if len(rust_values) > 0 else 0, True),
        (normalize_project(project_name), timestamp, np.mean(other_values) if len(other_values) > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [09:27<00:00, 20.89it/s]


In [10]:
metrics = TimestampMetric("rca_cognitive_complexity_per_unit", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_values = []
    other_values = []

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "FUNCTION":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_values.append(int(find_measurement(entry, "COGNITIVE", "sum") or 0))
        else:
            other_values.append(int(find_measurement(entry, "COGNITIVE", "sum") or 0))

    return [
        (normalize_project(project_name), timestamp, np.mean(rust_values) if len(rust_values) > 0 else 0, True),
        (normalize_project(project_name), timestamp, np.mean(other_values) if len(other_values) > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [10:09<00:00, 19.48it/s]


In [4]:
metrics = TimestampMetric("rca_loc", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_value = 0
    other_value = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_value += int(find_measurement(entry, "LOC", "ploc") or 0)
        else:
            other_value += int(find_measurement(entry, "LOC", "ploc") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_value, True),
        (normalize_project(project_name), timestamp, other_value, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [10:49<00:00, 18.26it/s]


In [11]:
metrics = TimestampMetric("rca_loc_per_unit", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_values = []
    other_values = []

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "FUNCTION":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_values.append(int(find_measurement(entry, "LOC", "ploc") or 0))
        else:
            other_values.append(int(find_measurement(entry, "LOC", "ploc") or 0))

    return [
        (normalize_project(project_name), timestamp, np.mean(rust_values) if len(rust_values) > 0 else 0, True),
        (normalize_project(project_name), timestamp, np.mean(other_values) if len(other_values) > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:06<00:00, 24.37it/s]


In [12]:
metrics = TimestampMetric("rca_test_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_loc = 0
    other_loc = 0
    rust_tloc = 0
    other_tloc = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "UNIT":
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            rust_tloc += int(find_measurement(entry, "LOC", "tloc") or 0)
        else:
            other_loc += int(find_measurement(entry, "LOC", "ploc") or 0)
            other_tloc += int(find_measurement(entry, "LOC", "tloc") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_tloc / rust_loc if rust_loc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_tloc / other_loc if other_loc > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [09:03<00:00, 21.82it/s]


In [13]:
metrics = TimestampMetric("rca_assertions_complexity_ratio", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)

    rust_assertions = 0
    other_assertions = 0
    rust_complexity = 0
    other_complexity = 0

    for entry in get_zip_json_data(zip_file):
        if entry["qualifier"] != "FUNCTION":
            continue

        if entry["language"] == "rust" and phase != "pre":
            rust_assertions += int(find_measurement(entry, "THESIS", "assertions") or 0)
        else:
            other_assertions += int(find_measurement(entry, "THESIS", "assertions") or 0)

        tloc = int(find_measurement(entry, "LOC", "tloc") or 0)
        if tloc > 0:
            continue
        if entry["language"] == "rust" and phase != "pre":
            rust_complexity += int(find_measurement(entry, "CYCLOMATIC", "sum") or 0)
        else:
            other_complexity += int(find_measurement(entry, "CYCLOMATIC", "sum") or 0)

    return [
        (normalize_project(project_name), timestamp, rust_assertions / rust_complexity if rust_complexity > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_assertions / other_complexity if other_complexity > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json.zip", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 11861/11861 [08:55<00:00, 22.16it/s]
